In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, IndexToString
from pyspark.ml.classification import RandomForestClassifier

# 1. Load the table first
raw_data = spark.table("instagram.goldlayer.vw_behavioral_user_insights")

# 2. Prep columns - fixed the 'df' reference error here
df_prepped = raw_data.withColumn("biometric_login_num", (raw_data["biometric_login_used"] == "Yes").cast("int"))

# 3. Index the labels (mapping strings like "High" to 0.0)
l_indexer = StringIndexer(inputCol="stress_category", outputCol="label").fit(df_prepped)
indexed_df = l_indexer.transform(df_prepped)

# 4. Assemble features into a vector
assembler = VectorAssembler(
    inputCols=['age', 'user_engagement_score', 'linked_accounts_count', 'biometric_login_num'], 
    outputCol="features")
final_input = assembler.transform(indexed_df)

# 5. Fit the Model 
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100)
model = rf.fit(final_input)

# 6. Predict and Map numbers back to "High/Low" labels
predictions = model.transform(final_input)

converter = IndexToString(inputCol="prediction", outputCol="pred_stress_category", labels=l_indexer.labels)
final_output = converter.transform(predictions)

# 7. Save to Delta Table
final_output.select("user_id", "age", "stress_category", "pred_stress_category") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true")\
    .saveAsTable("instagram.model_output.ml_stress")

print("✅ Data loaded, model trained, and Delta table updated, bro!")